# Dynamic Straight Beam
Needs-Work

In [ ]:
#Bryson Mariano
#6/3/2026
#########################################################################################
#Libraaries
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
#from matplotlib.colors import LogNorm
#from matplotlib.colors import BoundaryNorm

#########################################################################################
#Universal Parameters
me    = 9.1093837 * 10 ** (-31)  #kg
c     = 299792458.0              #m/s
ne    = 1 * 10.0**15 *10**6      #number density
e0    = 8.85418782 * 10**(-12)   #m^-3 kg^-1s^4 A^2
e     = 1.60217663 * 10**(-19)   #coulombs
wp    = ((ne*e**2)/(me*e0)) ** (1/2)

#Define Parameters
gamma = 110                      #Unitless
k     = 0.475 *me*wp**2          #kg/s^2
rb    = 0.65  *c/wp              #c/omega_p   
px    = 110   *me*c              #mec

f0=((px)**2) / (gamma * me * k)
#########################################################################################
#Adjust these parameters 
y0rb= 0.3
dyrb= 0.01

epsilon=1e-3

y0= y0rb
dy= dyrb

Ymid=(y0+dy)/2
#########################################################################################
#Speacial Functions
def f_cyl(i):
    return ( f0/(2*rb) *1/(np.sqrt(1-(i)**2)))

def f_cylprime(i):
    return ( f0 /(2*rb) * (1-(i)**2)**(-3/2) * (i**2)  )

#Compute out repetative terms
f1=f_cyl(y0)
f2=f_cyl(y0+dy)
f3=f_cyl(Ymid)
#########################################################################################
#Define grid variables 
x_val=np.linspace(-0.05,0.1,100)
x=np.linspace(-0.05,0.1,250)
y=np.linspace(-0.6,y0*1.05,250)

x,y=np.meshgrid(x,y)

#Define New Parameters and formulas
sigma_x = 0.005           #sets width for x
           #sets height for x

fig, (ax1, ax2)= plt.subplots(1,2, figsize=(12,6))

ax2.axvline(x=0, linestyle='--')

im = ax2.pcolormesh(x, y, np.zeros_like(x), cmap='plasma', shading='auto',vmin=0, vmax=500)
cbar = fig.colorbar(im, ax=ax2, label='Electron Concentration')

def update(frame):

    mu_x = x_val[frame]

    if mu_x <0:
        sigma_y =np.abs(dy)       #sets Bandwidth for y
        mu_y    =Ymid
    else:
        sigma_y =np.abs(dy-((y0+dy)/f2-y0/f1)*mu_x)+epsilon      #sets Bandwidth for y
        mu_y    =-(Ymid/f3)*mu_x+Ymid                     #Sets Initial Height on Wakefield

    #Equations
    Phi_y = (1/(sigma_y*np.sqrt(2*math.pi)))   *    np.exp(-0.5*(y-mu_y)**2/sigma_y**2)    

    Phi=Phi_y

    #Define full-width half-max 
    FWHM = 2*np.sqrt(2*np.log(2))*sigma_y
    
    #Plots
    im.set_array(Phi.ravel())
    
    ax1.clear()
    #Plot1 - Y gaussian Distribution
    ax1.plot(Phi_y, y, color='blue', label="Prob. Dist.")
    #ax1.hlines(y=1/(2*sigma*np.sqrt(2*math.pi)), xmin=mu-sigma, xmax=mu+sigma, color='black', linestyle='--')

    #Set Limits and labels on x
    ax1.set_xlim(0,50)
    ax1.set_xlabel("phi")

    #Set Limits and labels on y
    ax1.set_ylim(-0.6,y0*1.05)
    ax1.set_ylabel("y")

    #Show the plot and the specific details of the plot
    ax1.text(0.05, 0.99, f"FWHM={FWHM}",
            transform=ax1.transAxes,
            fontsize=12,
            verticalalignment='top')
    return [im]


ani = FuncAnimation(
    fig,
    update,
    frames=len(x_val),
    interval=100,
    repeat=True
)

ani.save("electron_beam_straight.gif", writer="pillow", fps=10)

plt.close()
HTML(ani.to_jshtml())

